# Experiment: Case 001 High HbF Genotype Evidence Gap

Objective: convert the de-identified infancy hemoglobin record into a genotype-evidence gap map without diagnosing or recommending treatment.

Success criteria:
- keep the current label as `phenotype_only`;
- identify which explanations are plausible enough to ask a hematologist about;
- separate source-backed clinical interpretation from Nakafa Lab research hypotheses;
- produce a compact record-request list that does not include identifiers.


In [ ]:
from __future__ import annotations

from dataclasses import dataclass


@dataclass(frozen=True)
class Observation:
    """De-identified phenotype clue from the committed case-001 record."""

    key: str
    value: str
    interpretation: str


@dataclass(frozen=True)
class Hypothesis:
    """Potential explanation that still needs clinician or molecular confirmation."""

    name: str
    support: tuple[str, ...]
    blockers: tuple[str, ...]
    next_evidence: tuple[str, ...]
    research_use: str


def triage_score(hypothesis: Hypothesis) -> int:
    """Rank hypotheses by urgency to resolve, not by diagnostic probability."""
    return (10 * len(hypothesis.blockers)) - len(hypothesis.support)


def format_evidence(items: tuple[str, ...]) -> str:
    """Join evidence items into a short markdown-friendly sentence fragment."""
    if not items:
        return "none recorded"
    return "; ".join(items)

## Inputs

These inputs are copied only from the committed de-identified extraction. Raw medical PDFs stay outside Git.


In [ ]:
observations = [
    Observation(
        key="age_context",
        value="infancy, approximately 11 months",
        interpretation=(
            "near the age boundary where molecular testing remains important "
            "for beta-plus interpretation"
        ),
    ),
    Observation(
        key="severe_anemia",
        value="Hb 3.8-3.9 g/dL with microcytosis and hypochromia",
        interpretation=(
            "supports a severe thalassemia phenotype but does not define genotype"
        ),
    ),
    Observation(
        key="hemoglobin_f",
        value="HbF 97.6%",
        interpretation="directly relevant to HbF and HPFH-like rescue lanes",
    ),
    Observation(
        key="hemoglobin_a_present",
        value="hemoglobin types A, F, and A2 reported",
        interpretation=(
            "may indicate residual beta-globin production or transfusion "
            "context; needs source clarification"
        ),
    ),
    Observation(
        key="hemoglobin_a2",
        value="HbA2 2.0%",
        interpretation=(
            "does not by itself fit the classic age-over-12-month "
            "beta-thalassemia major pattern; needs molecular review"
        ),
    ),
    Observation(
        key="report_differential",
        value=(
            "beta-thalassemia homozygous or compound heterozygous; "
            "beta-plus suspected; HPFH or alpha-thalassemia "
            "co-inheritance possible"
        ),
        interpretation=(
            "the original report already points to a genotype-resolution gate"
        ),
    ),
]

[(item.key, item.value) for item in observations]

## Hypothesis Gate

The score below is a record-request priority. A higher score means more missing evidence must be resolved before patient-relevance claims.


In [ ]:
hypotheses = [
    Hypothesis(
        name="Severe beta-thalassemia with beta-plus involvement",
        support=(
            "severe early anemia",
            "HbA reported present",
            "original report suspected beta-plus",
        ),
        blockers=(
            "no HBB molecular result",
            "unknown transfusion proximity before historical HPLC",
        ),
        next_evidence=(
            "HBB sequencing or panel",
            "ask whether the historical hemoglobin analysis was pre-transfusion",
        ),
        research_use=(
            "keeps HbF rescue and beta-globin-restoration benchmarks "
            "relevant, with caveats"
        ),
    ),
    Hypothesis(
        name="Beta-thalassemia plus HPFH, delta-beta, or HBG modifier context",
        support=(
            "HbF 97.6%",
            "original report raised HPFH possibility",
        ),
        blockers=(
            "no HBG promoter or deletional HPFH result",
            "no F-cell distribution",
            "unknown adult baseline HbF off transfusion",
        ),
        next_evidence=(
            "HBG1/HBG2 promoter or HPFH/deletion testing if indicated",
            "F-cell or HbF distribution record if available",
        ),
        research_use=(
            "strengthens the HPFH-like rescue benchmark but does not "
            "prove the patient has HPFH"
        ),
    ),
    Hypothesis(
        name="Alpha-globin modifier or alpha copy-number context",
        support=(
            "original report raised alpha-thalassemia co-inheritance",
            "globin-chain balance is core beta-thalassemia biology",
        ),
        blockers=(
            "no HBA1/HBA2 deletion or duplication result",
            "no alpha/non-alpha globin balance readout",
        ),
        next_evidence=(
            "HBA1/HBA2 deletion and duplication testing",
            "ask if alpha copy-number changes affect interpretation",
        ),
        research_use=(
            "supports the alpha-globin burden and autophagy-cleanup lane "
            "as a testable assay question"
        ),
    ),
    Hypothesis(
        name="Transfusion-confounded hemoglobin fractions",
        support=("HbA reported present despite severe phenotype",),
        blockers=(
            "no transfusion-proximity note",
            "no repeat molecular confirmation",
        ),
        next_evidence=(
            "transfusion history around the historical HPLC",
            "molecular testing less affected by transfusion status",
        ),
        research_use="blocks overinterpretation of HbA, HbA2, and HbF",
    ),
]

ranked = sorted(hypotheses, key=triage_score, reverse=True)
[(item.name, triage_score(item), item.next_evidence) for item in ranked]

## Compact Record Request

This output is designed for the repo and the hematologist question sheet. It is not a diagnosis.


In [ ]:
record_request = {
    "current_case_label": "phenotype_only",
    "highest_value_missing_records": [
        "historical HPLC transfusion proximity",
        "HBB molecular result with beta-zero or beta-plus interpretation",
        "HBA1/HBA2 deletion and duplication result",
        "HPFH, delta-beta, or HBG promoter/modifier result if indicated",
        (
            "current pre-transfusion Hb, antibody screen, DAT, spleen "
            "status, iron MRI, and chelation packet"
        ),
    ],
    "research_boundary": (
        "case-001 supports HbF and globin-balance research relevance, "
        "but no patient-fit claim until molecular context is known"
    ),
}

record_request

## Durable Conclusion

Case-001 should remain `phenotype_only`. The historical record is strong enough to justify a genotype-first clinician question, especially because HbF was extremely high and HbA was reported present, but it is not enough to assign a molecular subtype.

Research consequence: the case strengthens the priority of HbF/HPFH-like rescue plus globin-balance readouts. It does not turn any candidate into a treatment recommendation.
